# Sample articles for evaluation

In [1]:
import os
import pandas as pd
tgt_lang = 'fr'
input_data_dir = '/Users/llupo/dev/wiki_mt/data/'
input_data_langdir = os.path.join(input_data_dir, f'parallel_corpora_enriched_split/en/{tgt_lang}/')
treatment_group_data_rebalanced_filename = os.path.join(input_data_langdir, 'treatment_group_rebalanced_after_sent_div_06B_d30.csv')

df_treat_group = pd.read_csv(treatment_group_data_rebalanced_filename)
df_treat_group.head(3)

/Users/llupo/miniconda3/envs/mentalenv/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,translationId,sourceTitle,targetTitle,sourceLanguage,sourceRevisionId,targetRevisionId,targetLanguage,sourceURL,targetURL,publishedDate,stats,qid,country_prediction,main_topic,main_topic_level1,year,main_topic_cleaned
0,179082,Samuel Yates,Samuel Yates,en,704696710.0,127883636.0,fr,//en.wikipedia.org/wiki/Samuel Yates,//fr.wikipedia.org/wiki/Samuel Yates,20160716132239,"{'any': 0.7743396226415095, 'human': 0.6467924...",Q7412973,United States,Culture.Biography.Biography*,Culture,2016,Culture.Biography
1,217629,Encyclopaedia Biblica,Encyclopaedia Biblica,en,674514587.0,130800780.0,fr,//en.wikipedia.org/wiki/Encyclopaedia Biblica,//fr.wikipedia.org/wiki/Encyclopaedia Biblica,20161019084918,"{'any': 0.18297533810660302, 'human': 0.096579...",Q5375654,United Kingdom,Culture.Biography.Biography*,Culture,2016,Culture.Biography
2,177973,Julie von May (von Rued),Julie von May (von Rued),en,722695084.0,127809339.0,fr,//en.wikipedia.org/wiki/Julie von May (von Rued),//fr.wikipedia.org/wiki/Julie von May (von Rued),20160713130136,"{'any': 1.0354938271604939, 'human': 0.6280864...",Q1712215,Switzerland,Culture.Biography.Women,Culture,2016,Culture.Biography


In [2]:
# Sample articles
N = 3
df_sampled = df_treat_group.sample(n=N, random_state=42)
df_sampled_ids = set(df_sampled['translationId'].tolist())

In [3]:
# Retrieve revision contents for the sampled articles
import ijson

json_path = os.path.join(input_data_langdir, 'revision_history_contents.json')
id_field = 'translationId'  # or 'qid' if using qids

sample_contents = []

with open(json_path, 'rb') as f:
    for article in ijson.items(f, 'item'):
        id = article[id_field]
        if id not in df_sampled_ids:
            continue

        # Append to the list
        sample_contents.append({
            'translationId': id,
            'sourceTitle': article['sourceTitle'],
            'targetTitle': article['targetTitle'],
            'sourceLanguage': article['sourceLanguage'],
            'targetLanguage': article['targetLanguage'],
            'raw_source': article['d30']['sourceContent'],
            'raw_target': article['d30']['targetContent']
        })

print (f"Retrieved {len(sample_contents)} articles with raw content.")
sample_contents[:3]

Retrieved 3 articles with raw content.


[{'translationId': 143553,
  'sourceTitle': 'Snow Leopard Trust',
  'targetTitle': 'Snow Leopard Trust',
  'sourceLanguage': 'en',
  'targetLanguage': 'fr',
  'raw_source': '{{Infobox organization\n| name   = The Snow Leopard Trust\n| logo   = [[Image:ISLT Logo 72 dpi.png]]\n| type   = \n| founded_date      = 1981\n| founder           = [[Helen Elaine Freeman]]\n| location          = [[Seattle, Washington]]\n| origins           = \n| key_people        = \n| area_served       = [[China]], [[India]], [[Kyrgyz Republic]], [[Mongolia]] & [[Pakistan]] \n| product           =\n| focus             = Snow leopard conservation\n| method            = \n| revenue           = \n| endowment         = \n| num_volunteers    = \n| num_employees     = \n| num_members       = 2500\n| subsib            = \n| owner             = \n| Non-profit_slogan = Saving Snow Leopards for 28 years\n| homepage          = http://www.snowleopard.org\n| dissolved         = \n| footnotes         = \n}}\n\nThe \'\'\'Snow L

# Preprocess articles for InfoGap pipeline

In [4]:
#!/usr/bin/env python3

import re
from collections import deque

from mwparserfromhell import parse, wikicode
from mwparserfromhell.parser import ParserError
from nltk.tokenize import sent_tokenize
from hanlp.utils.rules import split_sentence

# --------------------------------------------------------------------------- #
# Auxiliary-section dictionaries                                              #
# create a mapping of auxiliary section names by language,# following Wikipedia's manual of style: https://LANGUAGE_CODE.wikipedia.org/wiki/Wikipedia:Manual_of_Style/Layout
# in Simple Chinese it's: https://zh.wikipedia.org/wiki/Wikipedia:%E6%A0%BC%E5%BC%8F%E6%89%8B%E5%86%8A/%E7%89%88%E9%9D%A2%E4%BD%88%E5%B1%80#%E9%99%84%E9%8C%84%E5%85%83%E7%B4%A0
# --------------------------------------------------------------------------- #

AUXILIARY_SECTIONS = {
    'en': ['See also', 'References', 'Notes', 'Footnotes', 'Works cited', 'Sources', 'Citations', 'Bibliography', 'Endnotes', 'Further reading', 'External links', 'Related articles'],
    'es': ['Véase también', 'Referencias', 'Notas', 'Bibliografía', 'Bibliografía utilizada', 'Bibliografía adicional', 'Enlaces externos'],
    'fr': ['Voir aussi', 'Voir également', 'Références', 'Notes et références', 'Notes', 'Annexes', 'Autres projets', 'Bibliographie', 'Liens externes', 'Articles connexes', 'Articles liés'],
    'it': ['Note', 'Bibliografia', 'Voci correlate', 'Altri progetti', 'Collegamenti esterni', 'Riferimenti', 'Fonti', 'Voci correlate', 'Altri progetti', 'Collegamenti esterni', 'Articoli correlati', 'Articoli collegati'],
    'zh': ['補充解釋','註釋','備註','內文來源','參考資料','來源列表','參考文獻','參考書籍','相關條目','參見','參看','延伸閱讀','外部連結','維基媒體姊妹計劃','补充解释','注释','备注','内文来源','来源列表','参考资料','参考文献','参考书籍','相关条目','参见','参看','延展阅读','外部链接','维基媒体姊妹项目'],
    'zh-tw':['補充解釋','註釋','備註','內文來源','參考資料','來源列表','參考文獻','參考書籍','相關條目','參見','參看','延伸閱讀','外部連結','維基媒體姊妹計劃','补充解释','注释','备注','内文来源','来源列表','参考资料','参考文献','参考书籍','相关条目','参见','参看','延展阅读','外部链接','维基媒体姊妹项目']
    # Add more languages as needed
}

LANG_MAP = {
    'en': 'english',
    'fr': 'french',
    'it': 'italian',
    'ru': 'russian',
    'zh': 'chinese',
    'zh-tw': 'chinese',  # Traditional Chinese
    'es': 'spanish'
}

# --------------------------------------------------------------------------- #
# Pre-compiled regexes                                                        #
# --------------------------------------------------------------------------- #

AUX_PATTERNS = {
    lang: [re.compile(r'\b' + re.escape(section) + r'\b', re.IGNORECASE)  # re.complie complies repeatedly used patterns
           for section in sections]    # r'\b' ensures whole word match  re.escape(section) escapes special characters in section names
    for lang, sections in AUXILIARY_SECTIONS.items()
}
_TABLE_RE           = re.compile(r'\{\|[\s\S]*?\n\|}', flags=re.DOTALL)
_PARAGRAPH_RE       = re.compile(r'\n\s*\n')   # blank line = new paragraph
_REF_RE             = re.compile(r'<ref[^>/]*?/?>.*?</ref\s*>', flags=re.DOTALL | re.IGNORECASE)
_SELFREF_RE         = re.compile(r'<ref[^>/]*/\s*>', flags=re.IGNORECASE)
_LIST_LINE_RE       = re.compile(r'^[ \t]*[*#;:][^:].*$', re.MULTILINE)
_LINE_NON_WORD_RE   = re.compile(r'^([^\w\s].*)$', re.MULTILINE)
_LINE_WORD_COL_RE   = re.compile(r'^\s*\w+[\|:].*?$',  re.MULTILINE)
_PAREN_COMMA_RE     = re.compile(r'[\(（][\s　]*[,，][\s　]*[\)）]')
_PAREN_EMPTY_RE     = re.compile(r'[\(（][\s　]*[\)）]')
_NLBREAK_RE         = re.compile(r'\n{3,}')
_MULTISPACE_RE      = re.compile(r' {2,}')
_INFOBOX_TABLE_RE   = re.compile(r'\{\|\s*class=.*?infobox.*?\|\}',
                                 re.DOTALL | re.IGNORECASE)
_MAPFRAME_RE        = re.compile(r'<mapframe.*?</mapframe>',
                                 re.DOTALL | re.IGNORECASE)
_GEOJSON_RE         = re.compile(r'\{["\']?type["\']?\s*:\s*["\']Feature["\'].*?\}',
                                 re.DOTALL)
_INFOBOX_RE         = re.compile(
                                r"\{\{[Ii]nfobox[\s\S]*?\}\}",  # non-greedy across lines
                                flags=re.MULTILINE)

# --------------------------------------------------------------------------- #
# Helper functions                                                            #
# --------------------------------------------------------------------------- #

def split_article(text: str,
                  language: str = 'en',
                  *,
                  into_sentences: bool = False):

    # 1) coarse paragraph split on blank lines ............................
    paragraphs = _PARAGRAPH_RE.split(text)
    paragraphs = [p.strip('\n') for p in paragraphs if p.strip()]

    if not into_sentences:
        return paragraphs

    # 3) sentence segmentation ...........................................
    sentences = []
    is_chinese = language in ('zh', 'zh-tw')

    for p in paragraphs:

        if is_chinese:
            sentences.extend(split_sentence(p))
        else:
            nltk_lang = LANG_MAP.get(language, language)
            sentences.extend(sent_tokenize(p, language=nltk_lang))

    return [s.strip() for s in sentences if s.strip()]


def detect_headers(text: str) -> bool:
    """
    Return True if `text` *starts* with a MediaWiki section header like
    '== References ==' (trailing content on the same line is ignored).
    """
    return bool(re.match(r'^={2,}\s*[^=].*?[^=]\s*={2,}', text.lstrip()))


def remove_infobox(wikitext: str) -> str:
    """
    Remove all *top-level* infobox templates from a block of Wikitext.

    The function attempts a structured removal using **mwparserfromhell**.
    If that fails (e.g., malformed markup), it falls back to a conservative
    regex that deletes the first `{{Infobox …}}`–`}}` pair it encounters.

    Parameters
    ----------
    wikitext : str
        Raw Wikitext page contents.

    Returns
    -------
    str
        Wikitext with infobox templates stripped out.
    """
    try:
        code = parse(wikitext)

        # Remove every template whose canonical name contains "infobox"
        removed_any = False
        for tmpl in code.filter_templates():
            if "infobox" in tmpl.name.strip_code().lower():
                code.remove(tmpl)
                removed_any = True

        if removed_any:
            text = str(code)
        else:
            # Fallback (or nothing removed)
            text = _INFOBOX_RE.sub("", wikitext)

    except (ParserError, ValueError) as parse_err:
        # Fallback (or nothing removed)
        text = _INFOBOX_RE.sub("", wikitext)

    # Regex fallback for any remaining infoboxes or related markup
    text = _INFOBOX_TABLE_RE.sub('', text) # Remove infobox tables
    text = _MAPFRAME_RE.sub('', text) # Remove mapframe elements
    text = _GEOJSON_RE.sub('', text) # Remove geojson elements
    return text

def remove_auxiliary_sections(text: str,
                              language: str = "en",
                              remove_headers: bool = True) -> str:
    """
    Strip everything from the first auxiliary-section header onward and, if
    requested, drop *all* ordinary section headers (including nested ones)
    that precede it.
    """
    # -- Defensive guard --------------------------------------------------
    try:
        aux_patterns = AUX_PATTERNS[language]
    except KeyError:                         # keep the original ValueError style
        raise ValueError(f"Language {language!r} not supported")

    # -- 1) Split on blank lines -----------------------------------------
    segments = deque(split_article(text, language=language,
                                   into_sentences=False))

    result_segments: list[str] = []

    # -- 2) Consume segments until we meet an aux. header ----------------
    while segments:
        segment: str = segments.popleft().lstrip()
        if not segment:        # skipped empty paragraph
            continue

        # 2a) Repeatedly peel any leading headers inside *this* paragraph
        while detect_headers(segment):
            header_line, *rest = segment.split('\n', 1)

            # Auxiliary header?  ->  bail out completely
            if any(p.search(header_line) for p in aux_patterns):
                return '\n\n'.join(result_segments)

            if not remove_headers:          # keep ordinary header if wanted
                result_segments.append('# ' + header_line)

            segment = rest[0].lstrip() if rest else ''
            if not segment:                 # header-only paragraph
                break                       # → fetch next segment

        # 2b) After peeling, if something remains, keep it
        if segment:
            result_segments.append(segment)

    # Exhausted all segments without hitting an auxiliary header
    return '\n\n'.join(result_segments)

# --------------------------------------------------------------------------- #
# Main function                                                               #
# --------------------------------------------------------------------------- #

def clean_extracts(text, remove_headers=True, language='en', into_sentences = False):
    """
    Clean Wikipedia text where headers are simple short sentences (≤ 5 words)
    and remove infobox remnants and other markup elements.
    
    Args:
        text (str): The Wikipedia text to clean
        remove_headers (bool, optional): Whether to remove all short sentences that could be headers.
        language (str): Language code for auxiliary section removal.
                        Open to Single Language and Language Pairs
    
    Returns:
        str: The cleaned text
    """
    if not text:
        return ""

    if language not in AUXILIARY_SECTIONS:
        raise ValueError(f"Language {language} not supported")
    
    # Kill *all* wikitables
    text = _TABLE_RE.sub('', text)
    
    # Remove infobox before parsing  like {| class="infobox vcard" <mapframe>...</mapframe>
    text = remove_infobox(text)

    # 3. references  ←  ADD THESE TWO LINES
    text = _SELFREF_RE.sub('', text)
    text = _REF_RE.sub('', text)

    # Remove all lists
    text = _LIST_LINE_RE.sub('', text)
    
    # Process paragraphs until we hit an auxiliary section
    text = remove_auxiliary_sections(text, language=language, remove_headers=remove_headers)

    # Remove MediaWiki markup
    text = parse(text)
    text = wikicode.Wikicode.strip_code(text)

    # Additional regex cleaning
    text = _LINE_NON_WORD_RE.sub('', text) # Remove every line
    text = _LINE_WORD_COL_RE.sub('', text) # Remove lines that start with a word followed by a colon
    text = _PAREN_COMMA_RE.sub(' ', text) # Remove parentheses with a comma inside
    text = _PAREN_EMPTY_RE.sub(' ', text) # Remove parentheses with no content inside
    
    # Clean up whitespace
    text = _NLBREAK_RE.sub('\n\n', text)
    text = _MULTISPACE_RE.sub(' ', text)

    # Sentence segmentation if required (again because earlier we might have encounter)
    if into_sentences:
        segments = split_article(text, language=language, into_sentences=into_sentences)
        text = '\n'.join(segments)
    
    return text.strip()

In [5]:
# clean raw_source and raw_target from sample_contents
item = sample_contents[0]
cleaned_en = clean_extracts(item['raw_source'], remove_headers=False, language='en', into_sentences=False)
cleaned_fr = clean_extracts(item['raw_target'], remove_headers=False, language='fr', into_sentences=False)
print(cleaned_en)

The Snow Leopard Trust is the largest and oldest organization working solely to protect the endangered snow leopard (Panthera uncia) and its habitat in 12 countries of Central Asia. The trust is a non-profit organization with its headquarters in Seattle, Washington. The present total population of snow leopards in the wild is estimated at between 3,920 and 6,390.

 == History ==

The trust was founded in 1981 by Helen Elaine Freeman (March 10, 1932 – September 20, 2007). Working as a volunteer at Seattle's Woodland Park Zoo, Freeman became fascinated with the snow leopards there and learnt about their endangered plight. She later joined the staff of the zoo and was motivated to set up the trust to protect the snow leopard in the wild, and its habitat. She also began the trust’s philosophy of helping the people sharing the snow leopard’s habitat improve their standard of living in exchange for protecting the animal.

 == Snow leopard conservation programs ==

The trust performs scientif

In [6]:
import os
import sys
import dill
import importlib
from pathlib import Path

REPO_ROOT = Path().resolve().parent
SRC_DIR = REPO_ROOT / "src"
for path in (SRC_DIR, REPO_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.chdir(REPO_ROOT)

if "flowmason" in sys.modules:
    importlib.reload(sys.modules["flowmason"])

from main_scrape_bios import process_wikipedia_text
from packages.constants import BIO_SAVE_DIR

def save_blocks(title, lang, text):
    blocks = process_wikipedia_text(text, lang)
    os.makedirs(BIO_SAVE_DIR, exist_ok=True)
    with open(f"{BIO_SAVE_DIR}/{title}_{lang}.pkl", "wb") as f:
        dill.dump(blocks, f)

# Persist cleaned blocks for each sampled article
for article in sample_contents:
    src_title = article['sourceTitle']
    tgt_title = article['sourceTitle']
    src_lang = article['sourceLanguage']
    tgt_lang_article = article['targetLanguage']
    cleaned_src = clean_extracts(article['raw_source'], remove_headers=False, language=src_lang, into_sentences=False)
    cleaned_tgt = clean_extracts(article['raw_target'], remove_headers=False, language=tgt_lang_article, into_sentences=False)
    save_blocks(src_title, src_lang, cleaned_src)
    save_blocks(tgt_title, tgt_lang_article, cleaned_tgt)
    print(f"Saved {src_title}_{src_lang} and {tgt_title}_{tgt_lang_article}")

Cache file not found: /Users/anniewang/Desktop/infogap/scratch/full_cache_gpt_en_zh/9c6b24f5f9c3abf10d6c32902a4f82bc8bd17cb939651c5049de7504d7af0e3c


None
Saved Snow Leopard Trust_en and Snow Leopard Trust_fr
Saved German submarine V-80_en and German submarine V-80_fr
Saved Victoire Du Bois_en and Victoire Du Bois_fr


In [7]:
# print as a test
process_wikipedia_text(cleaned_en, 'en')

[{'paragraph': 'The Snow Leopard Trust is the largest and oldest organization working solely to protect the endangered snow leopard (Panthera uncia) and its habitat in 12 countries of Central Asia. The trust is a non-profit organization with its headquarters in Seattle, Washington. The present total population of snow leopards in the wild is estimated at between 3,920 and 6,390.'},
 {'header_1': 'history'},
 {'paragraph': "The trust was founded in 1981 by Helen Elaine Freeman (March 10, 1932 – September 20, 2007). Working as a volunteer at Seattle's Woodland Park Zoo, Freeman became fascinated with the snow leopards there and learnt about their endangered plight. She later joined the staff of the zoo and was motivated to set up the trust to protect the snow leopard in the wild, and its habitat. She also began the trust’s philosophy of helping the people sharing the snow leopard’s habitat improve their standard of living in exchange for protecting the animal."},
 {'header_1': 'snow leop

# Post-pipeline analysis

In [156]:
import os
import dill
import json
import pandas as pd

outputs_dir = "/Users/llupo/dev/infogap/outputs/en_fr_gpt_logs"

# Load each file in the outputs_dir
outputs = []
filenames = os.listdir(outputs_dir)
for filename in filenames:
    with open(os.path.join(outputs_dir, filename), "rb") as fh:
        outputs.append(json.loads(fh.read().strip()))

cache_paths = []
for output in outputs:
    cache_paths.append(output[-2][1]['cache_path'])

output_dfs = []

for cache_path in cache_paths:
    with open(cache_path, "rb") as fh:
        df = dill.load(fh)
        # convert to pandas DataFrame for easier viewing
        try:
            df = df.to_pandas()
        except AttributeError:
            pass
        output_dfs.append(df)

output_dfs

[                                                  fact  fact_index  \
 0    These conservationists and educators often hav...         161   
 1    These microsatellite loci have enough variatio...         107   
 2    The study is conducted through the trust's Ind...         101   
 3    The support is for projects that meet the need...         163   
 4    In 2013, the Snow Leopard Trust was a key tech...         118   
 ..                                                 ...         ...   
 169  Freeman learned about the endangered plight of...          11   
 170  The trust will launch a long-term snow leopard...          99   
 171  The present total population of snow leopards ...           5   
 172  Conservation programs must meet four important...          43   
 173  SLIMS facilitates knowledge sharing around the...         117   
 
             person_name                              fact_aligned_sentence  \
 0    Snow Leopard Trust  Conservationists and educators working on

In [155]:
outputs[0][-2][1]['cache_path']

'/Users/llupo/dev/infogap/scratch/full_cache_gpt_en_fr/b204c338b3985e301cc45b222d49a59a3bf5f6492f15cc17df13894cca70b498'

In [128]:
import os
from pathlib import Path
import dill
import pandas as pd

scr = Path(os.environ["SCRATCH_DIR"])
cache = scr / "full_cache_gpt_en_fr"  # adjust language suffix
hash_id = "b204c338b3985e301cc45b222d49a59a3bf5f6492f15cc17df13894cca70b498"  # from the log line
with (cache / f"{hash_id}").open("rb") as fh:
    df = dill.load(fh)
    # convert to pandas DataFrame for easier viewing
    df = df.to_pandas()

# load corresponding pickle file
src_lang = df.language.unique()[0]
tgt_lang = df.language.unique()[1]
title_src = 'Snow Leopard Trust'
title_tgt = 'Conservation du léopard des neiges'
filename_src = f"wiki_food/{title_src}_en.pkl"
filename_tgt = f"wiki_food/{title_tgt}_fr.pkl"

df_src = pd.read_pickle((scr / filename_src).open("rb"))
df_tgt = pd.read_pickle((scr / filename_tgt).open("rb"))

# print whole src and tgt texts
src_txt = "\n".join([block[list(block.keys())[0]] for block in df_src])
tgt_txt = "\n".join([block[list(block.keys())[0]] for block in df_tgt])

print(f"-----{len(src_txt)} chars-----")
print(src_txt)
print(f"-----{len(tgt_txt)} chars-----")
print(tgt_txt)
print()

# print whole list of facts on both sides
src_texts = [row.fact for _, row in df[df.language == src_lang][df.person_name == title_src].iterrows()]
tgt_texts = [row.fact for _, row in df[df.language == tgt_lang][df.person_name == title_src].iterrows()]

print(f"-----{len(src_texts)} source facts-----")
for fact in src_texts:
    print(f"- {fact}")
print(f"-----{len(tgt_texts)} target facts-----")
for fact in tgt_texts:
    print(f"- {fact}")
    





-----10049 chars-----
The Snow Leopard Trust is the largest and oldest organization working solely to protect the endangered snow leopard (Panthera uncia) and its habitat in 12 countries of Central Asia. The trust is a non-profit organization with its headquarters in Seattle, Washington. The present total population of snow leopards in the wild is estimated at between 3,920 and 6,390.
history
The trust was founded in 1981 by Helen Elaine Freeman (March 10, 1932 – September 20, 2007). Working as a volunteer at Seattle's Woodland Park Zoo, Freeman became fascinated with the snow leopards there and learnt about their endangered plight. She later joined the staff of the zoo and was motivated to set up the trust to protect the snow leopard in the wild, and its habitat. She also began the trust’s philosophy of helping the people sharing the snow leopard’s habitat improve their standard of living in exchange for protecting the animal.
snow leopard conservation programs
The trust performs scie

/var/folders/hh/rjqw1xv93s516zyl87hymmr80000gn/T/ipykernel_24054/3370265936.py:36: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  src_texts = [row.fact for _, row in df[df.language == src_lang][df.person_name == title_src].iterrows()]
/var/folders/hh/rjqw1xv93s516zyl87hymmr80000gn/T/ipykernel_24054/3370265936.py:37: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  tgt_texts = [row.fact for _, row in df[df.language == tgt_lang][df.person_name == title_src].iterrows()]


In [127]:
df.person_name.unique

<bound method Series.unique of 0      Snow Leopard Trust
1      Snow Leopard Trust
2      Snow Leopard Trust
3      Snow Leopard Trust
4      Snow Leopard Trust
              ...        
169    Snow Leopard Trust
170    Snow Leopard Trust
171    Snow Leopard Trust
172    Snow Leopard Trust
173    Snow Leopard Trust
Name: person_name, Length: 174, dtype: object>

'paragraph'

In [63]:
df.language.unique()

array(['en', 'fr'], dtype=object)

In [ ]:
import os
import dill
import pandas as pd

outputs_path = "/Users/llupo/dev/infogap/outputs/en_fr_gpt_logs/run_0000.json"

outputs = []
with open(outputs_path, "r") as fh:
    for line in fh:
        payload = line.strip()
        if not payload:
            continue
        outputs.append(dill.loads(bytes.fromhex(payload)))

cache_path = "/Users/llupo/dev/infogap/scratch/full_cache_gpt_en_fr/07a4319f6553f5728294aa820b6fed210d164fc52fdaa206e4c53e9dcdc0d0fb"
with open(cache_path, "rb") as fh:
    df = dill.load(fh)
    try:
        df = df.to_pandas()
    except AttributeError:
        pass

df

'/Users/llupo/dev/infogap/scratch/full_cache_gpt_en_fr/bc564360baba819bfeb3e880fc0737c655c731bf1c1b566e2f4a6a36f50d282f'

In [ ]:
# load pkl
